In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Dict, Tuple, List, Optional
import numpy as np
from dataclasses import dataclass
from torchinfo import summary
import os

In [2]:
class ImageCNN(nn.Module):
    def __init__(self, output_dim=64, input_size=64):
        super(ImageCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1) 
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        
        self.batch_norm1 = nn.BatchNorm2d(32)
        self.batch_norm2 = nn.BatchNorm2d(64)
        self.batch_norm3 = nn.BatchNorm2d(128)
        self.batch_norm4 = nn.BatchNorm2d(256)
        
        final_size = input_size // 16
        self.fc1 = nn.Linear(256 * final_size * final_size, 512)
        self.fc_out = nn.Linear(512, output_dim) 
        
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.batch_norm1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm3(self.conv3(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm4(self.conv4(x))), 2)
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc_out(x)


class FullE2EModel(nn.Module):
    def __init__(self, basic_feature_dim, image_cnn_output_dim=64, basic_mlp_output_dim=32, input_grid_size=64):
        super(FullE2EModel, self).__init__()
        
        self.image_cnn = ImageCNN(output_dim=image_cnn_output_dim, input_size=input_grid_size)
        
        self.basic_mlp = nn.Sequential(
            nn.Linear(basic_feature_dim, basic_feature_dim * 2),
            nn.ReLU(),
            nn.BatchNorm1d(basic_feature_dim * 2),
            nn.Dropout(0.3),
            nn.Linear(basic_feature_dim * 2, basic_mlp_output_dim),
            nn.ReLU()
        )
        
        combined_dim = image_cnn_output_dim + basic_mlp_output_dim
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 1) 
        )
    
    def forward(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        output = self.head(combined) # 96차원 -> 1차원
        return output

    # 피처 추출을 위한 '머리 없는' forward
    def extract_features(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        return combined # 96차원 피처 반환

In [3]:
CNN_feature_extractor = FullE2EModel(basic_feature_dim=41)
print(CNN_feature_extractor)

FullE2EModel(
  (image_cnn): ImageCNN(
    (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (batch_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (batch_norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (batch_norm3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (batch_norm4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (fc1): Linear(in_features=4096, out_features=512, bias=True)
    (fc_out): Linear(in_features=512, out_features=64, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (basic_mlp): Sequential(
    (0): Linear(in_features=41, out_fe

In [4]:
p_value_input = torch.randn(32, 1, 64, 64)  # 배치 크기 32, 흑백 이미지 (1 채널), 64x64 크기
p_basic_input = torch.randn(32, 41)  # 배치 크기 32, 기본 피처 41개

dummy_dict_input = {
    'x_image': p_value_input,
    'x_basic': p_basic_input
}

summary(CNN_feature_extractor, input_data=dummy_dict_input, col_names=("input_size", "output_size", "num_params"))

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
FullE2EModel                             --                        [32, 1]                   --
├─ImageCNN: 1-1                          [32, 1, 64, 64]           [32, 64]                  --
│    └─Conv2d: 2-1                       [32, 1, 64, 64]           [32, 32, 64, 64]          320
│    └─BatchNorm2d: 2-2                  [32, 32, 64, 64]          [32, 32, 64, 64]          64
│    └─Conv2d: 2-3                       [32, 32, 32, 32]          [32, 64, 32, 32]          18,496
│    └─BatchNorm2d: 2-4                  [32, 64, 32, 32]          [32, 64, 32, 32]          128
│    └─Conv2d: 2-5                       [32, 64, 16, 16]          [32, 128, 16, 16]         73,856
│    └─BatchNorm2d: 2-6                  [32, 128, 16, 16]         [32, 128, 16, 16]         256
│    └─Conv2d: 2-7                       [32, 128, 8, 8]           [32, 256, 8, 8]           295,168
│    └─BatchNorm2d: